# Controlled Block-Encoding Test

This notebook constructs and tests `TwoParticleControlledSparseBlockEncoding` for a small demo tensor.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np

from qualtran.resource_counting import QECGatesCost, QubitCount, get_cost_value
from qualtran.resource_counting._call_graph import get_bloq_callee_counts

if (Path.cwd() / 'src').exists():
    REPO_ROOT = Path.cwd()
elif (Path.cwd().parent / 'src').exists():
    REPO_ROOT = Path.cwd().parent
else:
    REPO_ROOT = Path.cwd()

SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from exciton.benchmark_tensors import generate_v_tensor
from integrations.qualtran.block_encoding.two_particle_row_oracles import (
    build_two_particle_sparse_block_encoding,
    build_two_particle_controlled_sparse_block_encoding,
)


In [2]:
# Demo parameters
D = 1
L = 4
R_c = 1
R_loc = 1
entry_bitsize = 6

M = generate_v_tensor(
    L=L,
    D=D,
    r_c=R_c,
    r_loc=R_loc,
    metric='chebyshev',
    oracle_convention='direct',
)

base = build_two_particle_sparse_block_encoding(
    M=M, D=D, L=L, R_c=R_c, R_loc=R_loc, entry_bitsize=entry_bitsize
)
ctrl = build_two_particle_controlled_sparse_block_encoding(
    M=M, D=D, L=L, R_c=R_c, R_loc=R_loc, entry_bitsize=entry_bitsize
)

print('Base type:      ', type(base).__name__)
print('Controlled type:', type(ctrl).__name__)
print('Base signature:')
print(base.signature)
print('Controlled signature:')
print(ctrl.signature)


Base type:       TwoParticleSparseBlockEncoding
Controlled type: TwoParticleControlledSparseBlockEncoding
Base signature:
Signature((Register(name='q', dtype=QBit(), _shape=(), side=<Side.THRU: 3>), Register(name='m', dtype=BQUInt(bitsize=2, iteration_length=3), _shape=(1,), side=<Side.THRU: 3>), Register(name='l', dtype=BQUInt(bitsize=2, iteration_length=3), _shape=(1,), side=<Side.THRU: 3>), Register(name='i', dtype=BQUInt(bitsize=2, iteration_length=4), _shape=(1,), side=<Side.THRU: 3>), Register(name='j', dtype=BQUInt(bitsize=2, iteration_length=4), _shape=(1,), side=<Side.THRU: 3>)))
Controlled signature:
Signature((Register(name='ctrl', dtype=QBit(), _shape=(), side=<Side.THRU: 3>), Register(name='q', dtype=QBit(), _shape=(), side=<Side.THRU: 3>), Register(name='m', dtype=BQUInt(bitsize=2, iteration_length=3), _shape=(1,), side=<Side.THRU: 3>), Register(name='l', dtype=BQUInt(bitsize=2, iteration_length=3), _shape=(1,), side=<Side.THRU: 3>), Register(name='i', dtype=BQUInt(bitsiz

In [3]:
# Basic resource checks
base_qubits = int(get_cost_value(base, QubitCount()))
ctrl_qubits = int(get_cost_value(ctrl, QubitCount()))
print(f'Base logical qubits:       {base_qubits}')
print(f'Controlled logical qubits: {ctrl_qubits}')

print('Base QEC gate counts:')
print(get_cost_value(base, QECGatesCost()))

print('Controlled QEC gate counts (attempt):')
try:
    print(get_cost_value(ctrl, QECGatesCost()))
except Exception as exc:
    print('QECGatesCost failed for controlled bloq (expected current limitation):')
    print(type(exc).__name__, exc)

# Fallback structural sanity: call graph extraction while ignoring decomp failures
callees = get_bloq_callee_counts(ctrl, ignore_decomp_failure=True)
print(f'Controlled call-graph entries (ignore_decomp_failure=True): {len(callees)}')


Base logical qubits:       22
Controlled logical qubits: 23
Base QEC gate counts:
and_bloq: 187, clifford: 744, rotation: 14, measurement: 187
Controlled QEC gate counts (attempt):
QECGatesCost failed for controlled bloq (expected current limitation):
DecomposeTypeError Could not build call graph for C[And]: And is atomic.
Controlled call-graph entries (ignore_decomp_failure=True): 1


## Small-Scale Matrix Verification (Base Block-Encoding)

For a tiny instance, extract the dense matrix of `TwoParticleSparseBlockEncoding` and verify unitarity.

In [4]:
import cirq
from qualtran.cirq_interop import BloqAsCirqGate, get_named_qubits

def bloq_dense_unitary(bloq):
    gate = BloqAsCirqGate(bloq)
    named = get_named_qubits(bloq.signature.lefts())
    qorder = []
    for reg in bloq.signature.lefts():
        qorder.extend(np.array(named[reg.name]).reshape(-1).tolist())
    op = gate.on(*qorder)
    return cirq.unitary(op)


In [5]:
# Tiny instance for dense-matrix extraction
D_small = 1
L_small = 2
R_c_small = 1
R_loc_small = 1
entry_bitsize_small = 3

M_small = generate_v_tensor(
    L=L_small,
    D=D_small,
    r_c=R_c_small,
    r_loc=R_loc_small,
    metric='chebyshev',
    oracle_convention='direct',
)

base_small = build_two_particle_sparse_block_encoding(
    M=M_small,
    D=D_small,
    L=L_small,
    R_c=R_c_small,
    R_loc=R_loc_small,
    entry_bitsize=entry_bitsize_small,
)

U = bloq_dense_unitary(base_small)
nq = base_small.signature.n_qubits()
I = np.eye(U.shape[0], dtype=complex)
unitarity_err = np.max(np.abs(U.conj().T @ U - I))

print('Base-small type:', type(base_small).__name__)
print('n_qubits:', nq)
print('Unitary shape:', U.shape)
print('Max unitarity error ||U^\dagger U - I||_max:', unitarity_err)
print('Top-left 6x6 block (real part):')
print(np.real_if_close(U[:6, :6]))


Base-small type: TwoParticleSparseBlockEncoding
n_qubits: 7
Unitary shape: (128, 128)
Max unitarity error ||U^\dagger U - I||_max: 4.107825191113079e-15
Top-left 6x6 block (real part):
[[ 0.        +0.00000000e+00j  0.        +0.00000000e+00j
   0.        +0.00000000e+00j  0.        +0.00000000e+00j
   0.        +0.00000000e+00j  0.        +0.00000000e+00j]
 [ 0.08504076-4.25834634e-34j  0.11111111-4.25834634e-34j
   0.08504076-4.25834634e-34j  0.17008153+0.00000000e+00j
   0.02834692-4.00886000e-02j -0.07407407+1.04756560e-01j]
 [ 0.17008153+0.00000000e+00j  0.08504076-4.25834634e-34j
   0.11111111-4.25834634e-34j  0.08504076-4.25834634e-34j
   0.05669384-8.01772000e-02j -0.05669384+8.01772000e-02j]
 [ 0.        +0.00000000e+00j  0.        +0.00000000e+00j
   0.        +0.00000000e+00j  0.        +0.00000000e+00j
   0.        +0.00000000e+00j  0.        +0.00000000e+00j]
 [ 0.        +0.00000000e+00j  0.        +0.00000000e+00j
   0.        +0.00000000e+00j  0.        +0.00000000e+00j

### Note on Controlled Matrix Extraction
At present, dense matrix extraction of the controlled wrapper may fail in Qualtran because `Controlled[And]` tensorization has a known limitation for non-thru registers. The base block-encoding matrix extraction above is the reliable small-scale check.

In [6]:
# Reproducible matrix verification (same checks used in terminal)
import cirq
from qualtran.cirq_interop import BloqAsCirqGate, get_named_qubits

D_v, L_v, R_c_v, R_loc_v, entry_bitsize_v = 1, 2, 1, 1, 3
M_v = generate_v_tensor(
    L=L_v, D=D_v, r_c=R_c_v, r_loc=R_loc_v, metric='chebyshev', oracle_convention='direct'
)
bloq_v = build_two_particle_sparse_block_encoding(
    M=M_v, D=D_v, L=L_v, R_c=R_c_v, R_loc=R_loc_v, entry_bitsize=entry_bitsize_v
)

named_v = get_named_qubits(bloq_v.signature.lefts())
qorder_v = []
for reg in bloq_v.signature.lefts():
    qorder_v.extend(np.array(named_v[reg.name]).reshape(-1).tolist())

op_v = BloqAsCirqGate(bloq_v).on(*qorder_v)
U_v = cirq.unitary(op_v)

nq_v = bloq_v.signature.n_qubits()
expected_dim_v = 2 ** nq_v
shape_ok_v = U_v.shape == (expected_dim_v, expected_dim_v)

I_v = np.eye(expected_dim_v, dtype=complex)
unitarity_max_v = float(np.max(np.abs(U_v.conj().T @ U_v - I_v)))

rng = np.random.default_rng(7)
psi = rng.normal(size=expected_dim_v) + 1j * rng.normal(size=expected_dim_v)
psi = psi / np.linalg.norm(psi)

sim = cirq.Simulator(dtype=np.complex128)
res = sim.simulate(cirq.Circuit(op_v), initial_state=psi, qubit_order=qorder_v)
psi_sim = res.final_state_vector
psi_mat = U_v @ psi
state_err_v = float(np.max(np.abs(psi_sim - psi_mat)))

print({'D': D_v, 'L': L_v, 'R_c': R_c_v, 'R_loc': R_loc_v, 'entry_bitsize': entry_bitsize_v})
print('matrix_shape:', U_v.shape, 'expected_dim:', expected_dim_v, 'shape_ok:', shape_ok_v)
print('unitarity_max_abs:', unitarity_max_v)
print('state_action_max_abs_error:', state_err_v)
print('PASS_shape:', shape_ok_v)
print('PASS_unitary:', unitarity_max_v < 1e-10)
print('PASS_state_action:', state_err_v < 1e-10)


{'D': 1, 'L': 2, 'R_c': 1, 'R_loc': 1, 'entry_bitsize': 3}
matrix_shape: (128, 128) expected_dim: 128 shape_ok: True
unitarity_max_abs: 4.107825191113079e-15
state_action_max_abs_error: 4.652682298944613e-16
PASS_shape: True
PASS_unitary: True
PASS_state_action: True
